# A decision tree — pure NumPy

**Dataset:** `wine.csv` (178 rows, 13 chemical features, 3 classes).

A tree is a sequence of threshold questions. Here you implement the split search, recursion, prediction and a depth sweep yourself.

## 1. Load and split

In [3]:
import numpy as np

raw = np.genfromtxt('wine.csv', delimiter=',', names=True)
feature_names = list(raw.dtype.names[:-1])
X = np.column_stack([raw[name] for name in feature_names]).astype(float)
y = np.asarray(raw['class'], dtype=int)

rng = np.random.default_rng(13)
train_idx, test_idx = [], []
for label in range(3):
    idx = np.where(y == label)[0]
    rng.shuffle(idx)
    cut = int(0.75 * len(idx))
    train_idx.extend(idx[:cut])
    test_idx.extend(idx[cut:])
train_idx, test_idx = np.array(train_idx), np.array(test_idx)
Xtr, Xte, ytr, yte = X[train_idx], X[test_idx], y[train_idx], y[test_idx]
print('train:', len(ytr), '| test:', len(yte), '| classes:', np.bincount(ytr).tolist())

train: 133 | test: 45 | classes: [44, 53, 36]

## 2. Gini impurity and the best threshold split

In [5]:
def gini(labels):
    if len(labels) == 0:
        return 0.0
    probabilities = np.bincount(labels, minlength=3) / len(labels)
    return float(1.0 - np.sum(probabilities ** 2))

def best_split(X_data, labels):
    best = None
    parent = gini(labels)
    for feature in range(X_data.shape[1]):
        values = np.unique(X_data[:, feature])
        thresholds = (values[:-1] + values[1:]) / 2.0
        for threshold in thresholds:
            left = X_data[:, feature] <= threshold
            if left.sum() == 0 or (~left).sum() == 0:
                continue
            weighted = (left.sum() * gini(labels[left]) + (~left).sum() * gini(labels[~left])) / len(labels)
            gain = parent - weighted
            if best is None or gain > best['gain']:
                best = {'feature': feature, 'threshold': float(threshold), 'gain': float(gain)}
    return best

print('root gini:', round(gini(ytr), 3))
print('best root split:', best_split(Xtr, ytr))

root gini: 0.658
best root split: {'feature': 12, 'threshold': 755.0, 'gain': 0.2663088979161115}

## 3. Recursive tree builder and predictor

In [7]:
def majority(labels):
    return int(np.bincount(labels, minlength=3).argmax())

def build_tree(X_data, labels, depth=0, max_depth=4):
    if depth >= max_depth or len(np.unique(labels)) == 1 or len(labels) < 6:
        return {'leaf': majority(labels), 'n': len(labels)}
    split = best_split(X_data, labels)
    if split is None or split['gain'] <= 1e-12:
        return {'leaf': majority(labels), 'n': len(labels)}
    left = X_data[:, split['feature']] <= split['threshold']
    return {
        'feature': split['feature'], 'threshold': split['threshold'],
        'left': build_tree(X_data[left], labels[left], depth + 1, max_depth),
        'right': build_tree(X_data[~left], labels[~left], depth + 1, max_depth),
        'n': len(labels),
    }

def predict_one(node, row):
    while 'leaf' not in node:
        node = node['left'] if row[node['feature']] <= node['threshold'] else node['right']
    return node['leaf']

def predict(tree, X_data):
    return np.array([predict_one(tree, row) for row in X_data])

model = build_tree(Xtr, ytr, max_depth=4)
pred = predict(model, Xte)
print('test accuracy:', round(float(np.mean(pred == yte)), 3))
print('tree root:', model.get('feature'), '<=', round(model.get('threshold', 0), 2))

test accuracy: 0.956
tree root: 12 <= 755.0

## 4. Depth is a capacity choice

In [9]:
for depth in range(1, 9):
    tree = build_tree(Xtr, ytr, max_depth=depth)
    train_acc = float(np.mean(predict(tree, Xtr) == ytr))
    test_acc = float(np.mean(predict(tree, Xte) == yte))
    print(f'depth={depth}: train={train_acc:.3f} | test={test_acc:.3f}')

depth=1: train=0.699 | test=0.689
depth=2: train=0.917 | test=0.889
depth=3: train=0.985 | test=0.956
depth=4: train=0.985 | test=0.956
depth=5: train=0.985 | test=0.956
depth=6: train=0.985 | test=0.956
depth=7: train=0.985 | test=0.956
depth=8: train=0.985 | test=0.956

## 5. Count how often each feature is used

In [11]:
def count_features(node, counts):
    if 'leaf' in node:
        return
    counts[node['feature']] += 1
    count_features(node['left'], counts)
    count_features(node['right'], counts)

counts = np.zeros(X.shape[1], dtype=int)
count_features(model, counts)
for index in np.argsort(counts)[::-1][:6]:
    print(f'{feature_names[index]:>28}: {counts[index]} split(s)')

             color_intensity: 2 split(s)
                  flavanoids: 2 split(s)
                     proline: 1 split(s)
                   magnesium: 1 split(s)
                     alcohol: 1 split(s)
             proanthocyanins: 0 split(s)

## 6. Limitations and honest reporting

- A single tree is unstable: small data changes can change the root split.
- Gini gain favours features with many possible thresholds; this implementation does not correct for that.
- Depth and minimum leaf size are capacity controls. A perfect training score is not evidence of generalisation.
- Feature usage is not causality. A correlated feature can stand in for the real driver.

**Exercise:** add a minimum leaf-size rule and compare test accuracy at depth 3, 5 and 8. Then explain why the best depth on this split might not be the best depth on another split.